# 00c_dataset_eda_and_freeze

**Task ID:** `DATASET_EDA_FREEZE`  
**Runner:** `experiments.scripts.profile_dataset`

## Scientific responsibility

Generate validated data cards, statistics and figures; audit quality/leakage; freeze eligible IDs and splits.

## Required artifacts

- `dataset.feasibility_receipts`
- `dataset.acquisition_inventory`

## Produced artifacts

- `dataset.profile`
- `dataset.unit_profile`
- `dataset.statistics`
- `dataset.split_statistics`
- `dataset.quality_issues`
- `dataset.attrition`
- `dataset.leakage_audit`
- `dataset.figure_manifest`
- `dataset.frozen_manifest`
- `dataset.registry`
- `dataset.endpoint_matrix`
- `dataset.cross_dataset_comparison`
- `dataset.eda_report`

This notebook is a thin orchestrator. Model execution, metrics, schemas and
artifact validation live in versioned Python modules. It must run in a clean
kernel and receives its frozen profile/provenance through executor-controlled
environment variables.


In [ ]:
# Executor-controlled configuration; fail closed when run directly.
import json
import os
from pathlib import Path

NOTEBOOK_ID = '00c_dataset_eda_and_freeze'
TASK_ID = 'DATASET_EDA_FREEZE'
RUNNER_MODULE = 'experiments.scripts.profile_dataset'
REQUIRED_ENV = (
    "MLS_BENCHMARK_PROFILE",
    "MLS_RUN_NAMESPACE",
    "MLS_NOTEBOOK_ID",
    "MLS_CONFIG_SHA256",
    "MLS_ARTIFACT_ROOT",
    "MLS_RUN_ID",
    "MLS_SOURCE_SHA256",
    "MLS_DEPENDENCY_STATE_SHA256",
    "MLS_EVIDENCE_STATUS",
)
missing = [name for name in REQUIRED_ENV if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Missing executor-controlled variables: {missing}")
if os.environ["MLS_NOTEBOOK_ID"] != NOTEBOOK_ID:
    raise RuntimeError("Notebook ID does not match executor contract")

RUN_CONTEXT = {name: os.environ[name] for name in REQUIRED_ENV}
print(json.dumps({"task_id": TASK_ID, **RUN_CONTEXT}, indent=2))


In [ ]:
# Delegate all scientific computation to the registered runner.
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    RUNNER_MODULE,
    "--profile",
    RUN_CONTEXT["MLS_BENCHMARK_PROFILE"],
    "--run-namespace",
    RUN_CONTEXT["MLS_RUN_NAMESPACE"],
    "--evidence-status",
    RUN_CONTEXT["MLS_EVIDENCE_STATUS"],
    "--artifact-root",
    RUN_CONTEXT["MLS_ARTIFACT_ROOT"],
    "--notebook-id",
    NOTEBOOK_ID,
    "--run-id",
    RUN_CONTEXT["MLS_RUN_ID"],
    "--source-sha256",
    RUN_CONTEXT["MLS_SOURCE_SHA256"],
    "--config-sha256",
    RUN_CONTEXT["MLS_CONFIG_SHA256"],
    "--dependency-state-sha256",
    RUN_CONTEXT["MLS_DEPENDENCY_STATE_SHA256"],
]
completed = subprocess.run(command, check=False, text=True)
if completed.returncode != 0:
    raise RuntimeError(f"Runner failed with exit code {completed.returncode}")


In [ ]:
# Revalidate evidence before the executor can promote this run.
from benchmarks.scripts.execute_all_notebooks import validate_produced_artifacts

validation_context = {
    "run_id": RUN_CONTEXT["MLS_RUN_ID"],
    "status": RUN_CONTEXT["MLS_EVIDENCE_STATUS"],
    "source_sha256": RUN_CONTEXT["MLS_SOURCE_SHA256"],
    "config_sha256": RUN_CONTEXT["MLS_CONFIG_SHA256"],
    "dependency_state_sha256": RUN_CONTEXT["MLS_DEPENDENCY_STATE_SHA256"],
}
entry_contract = {"expected_schema_version": 1, "group": "00", "id": "00c_dataset_eda_and_freeze", "network_policy": "none", "path": "experiments/notebooks/00c_dataset_eda_and_freeze.ipynb", "produces": ["dataset.profile", "dataset.unit_profile", "dataset.statistics", "dataset.split_statistics", "dataset.quality_issues", "dataset.attrition", "dataset.leakage_audit", "dataset.figure_manifest", "dataset.frozen_manifest", "dataset.registry", "dataset.endpoint_matrix", "dataset.cross_dataset_comparison", "dataset.eda_report"], "profiles": ["smoke", "full"], "requires": ["dataset.feasibility_receipts", "dataset.acquisition_inventory"], "responsibility": "Generate validated data cards, statistics and figures; audit quality/leakage; freeze eligible IDs and splits.", "runner": "experiments.scripts.profile_dataset", "task_id": "DATASET_EDA_FREEZE", "timeout_seconds": 7200}
validation_errors = validate_produced_artifacts(
    entry_contract,
    Path(RUN_CONTEXT["MLS_ARTIFACT_ROOT"]),
    RUN_CONTEXT["MLS_BENCHMARK_PROFILE"],
    RUN_CONTEXT["MLS_RUN_NAMESPACE"],
    validation_context,
)
if validation_errors:
    raise RuntimeError("Produced evidence failed validation:\n- " + "\n- ".join(validation_errors))
print(f"Validated {len(entry_contract['produces'])} artifact contract(s)")
